# [9.1] Refusal Directions and Safe Steering - Solutions

Reference validation notebook for the safe refusal-direction and steering-control section.

<img src="../../instructions/assets/refusal_directions_safe_steering_validation_loop.svg" width="860">

This solution notebook runs the reference toy contract, then checks the committed CUDA report for the scoped GT-2 aggregate evidence.

In [1]:
import json
import sys
from pathlib import Path

chapter = "chapter9_alignment_interpretability"
section = "part1_refusal_directions_safe_steering"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_refusal_directions_safe_steering.tests as tests
import part1_refusal_directions_safe_steering.solutions as solutions

GT_TIER = "GT-2"
EXERCISE_ID = "9_1_refusal_directions_and_safe_steering"
EXPECTED_RUNTIME = "seconds for toy contracts; minutes for the CUDA real-model preflights"
REQUIRES_GPU = True

## Toy Contract

<details>
<summary>Expected output</summary>

Each visible test prints `All tests ... passed!` and the contract dictionary contains direction, score, separation, steering, capability, random-control, label-shuffle, and comparison entries.

</details>

<details>
<summary>Help - reading these tests</summary>

The toy contract is deliberately small. It checks implementation shape and sign conventions before the report checks real-model evidence.

</details>

In [2]:
tests.test_direction_smoke_test(solutions.direction_smoke_test)
tests.test_scores_smoke_test(solutions.scores_smoke_test)
tests.test_separation_smoke_test(solutions.separation_smoke_test)
tests.test_steering_smoke_test(solutions.steering_smoke_test)
tests.test_capability_smoke_test(solutions.capability_smoke_test)
tests.test_random_control_smoke_test(solutions.random_control_smoke_test)
tests.test_label_shuffle_smoke_test(solutions.label_shuffle_smoke_test)
tests.test_comparison_smoke_test(solutions.comparison_smoke_test)

All tests in `test_direction_smoke_test` passed!
All tests in `test_scores_smoke_test` passed!
All tests in `test_separation_smoke_test` passed!
All tests in `test_steering_smoke_test` passed!
All tests in `test_capability_smoke_test` passed!
All tests in `test_random_control_smoke_test` passed!
All tests in `test_label_shuffle_smoke_test` passed!
All tests in `test_comparison_smoke_test` passed!


In [3]:
contract = solutions.run_smoke_test(cpu=True)
assert contract["direction"] == [1.0, 0.0]
assert contract["separation"]["separates_refusal"]
assert contract["steering"]["changes_refusal_rate"]
assert contract["capability"]["degradation_small"]
assert contract["random_control"]["random_direction_fails"]
assert contract["label_shuffle"]["label_shuffle_fails"]
assert contract["comparison"]["best_method"] == "mean_difference"
contract

{'direction': [1.0, 0.0],
 'scores': [2.0, 0.5],
 'separation': {'refusal_mean_score': 2.5,
  'non_refusal_mean_score': 0.25,
  'margin': 2.25,
  'accuracy': 1.0,
  'separates_refusal': True},
 'steering': {'baseline_refusal_rate': 0.3333333432674408,
  'steered_refusal_rate': 0.6666666865348816,
  'refusal_rate_delta': 0.3333333432674408,
  'changes_refusal_rate': True},
 'capability': {'baseline_capability': 0.8500000238418579,
  'steered_capability': 0.800000011920929,
  'degradation': 0.050000011920928955,
  'degradation_small': True},
 'random_control': {'target_direction_delta': 0.4,
  'random_direction_delta': 0.05,
  'margin': 0.35000000000000003,
  'random_direction_fails': True},
 'label_shuffle': {'true_accuracy': 1.0,
  'shuffled_accuracy': 0.0,
  'accuracy_gap': 1.0,
  'true_margin': 2.649999998509884,
  'shuffled_margin': -2.649999998509884,
  'label_shuffle_fails': True},
 'comparison': {'method_scores': {'mean_difference': 0.95,
   'probe': 0.9,
   'sae_feature': 0.85,


## Signature Result

<img src="../../instructions/assets/refusal_directions_safe_steering_signature_result.svg" width="860">

<details>
<summary>Interpreting the result</summary>

The public GT-2 path is the signature evidence: Qwen2.5-0.5B-Instruct on `josephmayo/refusal-compliance-pairs`, 32 train and 32 held-out prompts, held-out accuracy `1.0`, margin `6.576968312263489`, layer sweep best layer `23`, PC1 variance `0.25265154242515564`, and aggregate-only completion evidence. Raw prompt and completion text are not saved.

</details>

In [4]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]

tests.test_committed_gpu_report_matches_refusal_direction_contract(report)
assert report["accepted"] and report["tests_passed"]
assert report["gt_tier"] == "GT-2"
assert gpu["cuda_available"]
assert gpu["torch_version"].endswith("+cu132")
assert gpu["cuda_version"] == "13.2"
assert gpu["gt2_refusal_direction_dataset_id"] == "josephmayo/refusal-compliance-pairs"
assert gpu["gt2_refusal_direction_heldout_accuracy"] == 1.0
assert gpu["gt2_refusal_direction_layer_sweep_best_layer"] == 23
assert gpu["gt2_refusal_direction_pc1_variance_fraction"] > 0
assert gpu["gt2_refusal_direction_projection_delta"] < 0
assert not gpu["gt2_refusal_direction_raw_prompt_text_saved"]
assert not gpu["gt2_refusal_direction_completion_text_saved"]
assert gpu["gt2_refusal_direction_aggregate_metrics_only"]

{k: gpu[k] for k in [
    "device",
    "torch_version",
    "cuda_version",
    "real_lm_category_heldout_accuracy",
    "instruction_refusal_allowed_add_delta",
    "instruction_refusal_projection_delta",
    "gt2_refusal_direction_heldout_accuracy",
    "gt2_refusal_direction_heldout_margin",
    "gt2_refusal_direction_layer_sweep_best_layer",
    "gt2_refusal_direction_pc1_variance_fraction",
    "gt2_refusal_direction_projection_delta",
    "gt2_refusal_direction_random_direction_margin_gap",
    "peak_vram_gb",
]}

All tests in `test_committed_gpu_report_matches_refusal_direction_contract` passed!


{'device': 'NVIDIA GeForce RTX 5090 Laptop GPU',
 'torch_version': '2.12.1+cu132',
 'cuda_version': '13.2',
 'real_lm_category_heldout_accuracy': 0.9375,
 'instruction_refusal_allowed_add_delta': 1.4752511978149414,
 'instruction_refusal_projection_delta': -2.9102389812469482,
 'gt2_refusal_direction_heldout_accuracy': 1.0,
 'gt2_refusal_direction_heldout_margin': 6.576968312263489,
 'gt2_refusal_direction_layer_sweep_best_layer': 23,
 'gt2_refusal_direction_pc1_variance_fraction': 0.25265154242515564,
 'gt2_refusal_direction_projection_delta': -0.8125,
 'gt2_refusal_direction_random_direction_margin_gap': 6.42253303527832,
 'peak_vram_gb': 1.9031662940979004}

## Limitations

- The safe behavioral completion diagnostic on sanitized meta-prompts is not GT-2-ready and should not be used as the signature behavioral claim.
- The public GT-2 completion rubric is aggregate and marker-based, not human-rated.
- The result supports a useful refusal control direction under this setup; it does not prove broad deployment safety or literal one-dimensional refusal.